# 🔍 Hyperparameter Identification using GridSearchCV & RandomizedSearchCV

This notebook uses **cross-validation search** to identify the best hyperparameters for 8 ML models — no hardcoding.

| Model | Search Method |
|---|---|
| Linear Regression | GridSearchCV |
| Decision Tree | GridSearchCV |
| Random Forest | RandomizedSearchCV |
| SVM | GridSearchCV |
| Neural Network (MLP) | RandomizedSearchCV |
| KNN | GridSearchCV |
| Gradient Boosting / XGBoost / LightGBM | RandomizedSearchCV |
| K-Means | Manual loop (no CV — unsupervised) |

## 📦 Install Dependencies

In [ ]:
!pip install scikit-learn xgboost lightgbm pandas --quiet

## 📥 Imports & Dataset

In [ ]:
import warnings
import numpy as np
import pandas as pd
from scipy.stats import randint, uniform

from sklearn.datasets import load_wine
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from xgboost import XGBClassifier
XGBOOST_AVAILABLE = True

from lightgbm import LGBMClassifier
LIGHTGBM_AVAILABLE = True

warnings.filterwarnings('ignore')
print("✅ All imports successful!")

## 📊 Load & Prepare Dataset
We use **Wine** (binary classification) for all supervised models, and its features for K-Means.

In [ ]:
data = load_wine()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Dataset     : Wine")
print(f"Features    : {X.shape[1]}")
print(f"Train size  : {X_train.shape[0]}")
print(f"Test size   : {X_test.shape[0]}")
print(f"Classes     : {data.target_names}")

## 🛠️ Helper: Display Search Results

In [ ]:
def show_results(model_name, search, X_test, y_test, search_type="GridSearchCV"):
    """Display best hyperparameters and CV results from a fitted search object."""
    best_params = search.best_params_
    best_cv_score = search.best_score_
    test_score = search.best_estimator_.score(X_test, y_test)

    print(f"\n--- {model_name} ({search_type}) ---")
    print(f"   Best CV Score : {best_cv_score:.4f}")
    print(f"   Test Accuracy : {test_score:.4f}")
    print(f"   Best Params   : {best_params}\n")

    cv_df = pd.DataFrame(search.cv_results_)
    top5 = (cv_df[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']]
            .sort_values('rank_test_score')
            .head(5)
            .reset_index(drop=True))
    top5.columns = ['Parameters', 'Mean CV Score', 'Std CV Score', 'Rank']
    top5['Mean CV Score'] = top5['Mean CV Score'].round(4)
    top5['Std CV Score']  = top5['Std CV Score'].round(4)
    
    print("Top 5 Configurations Evaluated:")
    print(top5.to_string(index=False))
    return best_params, test_score

---
## 1. 📈 Linear Regression — GridSearchCV
Linear Regression has few hyperparameters; we search over `fit_intercept` and `positive`.

In [ ]:
from sklearn.linear_model import Ridge

# Ridge Regression (adds regularization alpha — more meaningful to tune)
param_grid_lr = {
    'alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    'fit_intercept': [True, False],
    'solver': ['auto', 'svd', 'cholesky', 'lsqr'],
}

ridge = Ridge()
grid_lr = GridSearchCV(
    ridge,
    param_grid_lr,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)
grid_lr.fit(X_train_scaled, y_train)

show_results("Ridge Regression", grid_lr, X_test_scaled, y_test, "GridSearchCV")

---
## 2. 🌿 Decision Tree — GridSearchCV

In [ ]:
param_grid_dt = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_depth': [None, 3, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2'],
}

dt = DecisionTreeClassifier(random_state=42)
grid_dt = GridSearchCV(
    dt,
    param_grid_dt,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)
grid_dt.fit(X_train, y_train)

show_results("Decision Tree", grid_dt, X_test, y_test, "GridSearchCV")

---
## 3. 🌲 Random Forest — RandomizedSearchCV
Large search space → RandomizedSearchCV is more efficient.

In [ ]:
param_dist_rf = {
    'n_estimators': randint(50, 500),
    'max_depth': [None, 5, 10, 20, 30],
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False],
    'criterion': ['gini', 'entropy'],
}

rf = RandomForestClassifier(random_state=42)
random_rf = RandomizedSearchCV(
    rf,
    param_distributions=param_dist_rf,
    n_iter=30,
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=0
)
random_rf.fit(X_train, y_train)

show_results("Random Forest", random_rf, X_test, y_test, "RandomizedSearchCV")

---
## 4. 🎯 Support Vector Machine (SVM) — GridSearchCV

In [ ]:
param_grid_svm = {
    'C': [0.01, 0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
    'gamma': ['scale', 'auto'],
    'degree': [2, 3, 4],        # only used when kernel='poly'
}

svm = SVC(random_state=42)
grid_svm = GridSearchCV(
    svm,
    param_grid_svm,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)
grid_svm.fit(X_train_scaled, y_train)

show_results("Support Vector Machine (SVM)", grid_svm, X_test_scaled, y_test, "GridSearchCV")

---
## 5. 🧠 Neural Network (MLP) — RandomizedSearchCV

In [ ]:
param_dist_mlp = {
    'hidden_layer_sizes': [(50,), (100,), (100, 50), (128, 64), (64, 64, 32)],
    'activation': ['relu', 'tanh', 'logistic'],
    'solver': ['adam', 'sgd', 'lbfgs'],
    'alpha': uniform(0.0001, 0.1),
    'learning_rate': ['constant', 'adaptive', 'invscaling'],
    'learning_rate_init': [0.001, 0.01, 0.0001],
    'max_iter': [200, 500, 1000],
    'early_stopping': [True, False],
}

mlp = MLPClassifier(random_state=42)
random_mlp = RandomizedSearchCV(
    mlp,
    param_distributions=param_dist_mlp,
    n_iter=25,
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=0
)
random_mlp.fit(X_train_scaled, y_train)

show_results("Neural Network (MLP)", random_mlp, X_test_scaled, y_test, "RandomizedSearchCV")

---
## 6. 📍 K-Nearest Neighbors (KNN) — GridSearchCV

In [ ]:
param_grid_knn = {
    'n_neighbors': list(range(1, 21)),
    'weights': ['uniform', 'distance'],
    'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
    'p': [1, 2],             # 1 = Manhattan, 2 = Euclidean
    'leaf_size': [10, 20, 30, 50],
}

knn = KNeighborsClassifier()
grid_knn = GridSearchCV(
    knn,
    param_grid_knn,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)
grid_knn.fit(X_train_scaled, y_train)

show_results("K-Nearest Neighbors (KNN)", grid_knn, X_test_scaled, y_test, "GridSearchCV")

---
## 7. 🚀 Gradient Boosting — RandomizedSearchCV
### 7a. Sklearn GradientBoostingClassifier

In [ ]:
param_dist_gb = {
    'n_estimators': randint(50, 400),
    'learning_rate': uniform(0.01, 0.3),
    'max_depth': randint(2, 8),
    'subsample': uniform(0.5, 0.5),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['sqrt', 'log2', None],
}

gb = GradientBoostingClassifier(random_state=42)
random_gb = RandomizedSearchCV(
    gb,
    param_distributions=param_dist_gb,
    n_iter=25,
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=0
)
random_gb.fit(X_train, y_train)

show_results("Gradient Boosting (sklearn)", random_gb, X_test, y_test, "RandomizedSearchCV")

### 7b. XGBoost — RandomizedSearchCV

In [ ]:
if XGBOOST_AVAILABLE:
    param_dist_xgb = {
        'n_estimators': randint(50, 400),
        'learning_rate': uniform(0.01, 0.3),
        'max_depth': randint(2, 10),
        'subsample': uniform(0.5, 0.5),
        'colsample_bytree': uniform(0.5, 0.5),
        'reg_alpha': uniform(0, 1),
        'reg_lambda': uniform(0.5, 2),
        'gamma': uniform(0, 0.5),
        'min_child_weight': randint(1, 10),
    }

    xgb = XGBClassifier(verbosity=0, use_label_encoder=False,
                        eval_metric='logloss', random_state=42)
    random_xgb = RandomizedSearchCV(
        xgb,
        param_distributions=param_dist_xgb,
        n_iter=25,
        cv=5,
        scoring='accuracy',
        random_state=42,
        n_jobs=-1,
        verbose=0
    )
    random_xgb.fit(X_train, y_train)
    show_results("XGBoost", random_xgb, X_test, y_test, "RandomizedSearchCV")
else:
    print(" XGBoost not installed. Run: pip install xgboost")

### 7c. LightGBM — RandomizedSearchCV

In [ ]:
if LIGHTGBM_AVAILABLE:
    param_dist_lgbm = {
        'n_estimators': randint(50, 400),
        'learning_rate': uniform(0.01, 0.3),
        'num_leaves': randint(20, 150),
        'max_depth': [-1, 5, 10, 20],
        'min_child_samples': randint(5, 50),
        'subsample': uniform(0.5, 0.5),
        'colsample_bytree': uniform(0.5, 0.5),
        'reg_alpha': uniform(0, 1),
        'reg_lambda': uniform(0, 1),
    }

    lgbm = LGBMClassifier(random_state=42, verbose=-1)
    random_lgbm = RandomizedSearchCV(
        lgbm,
        param_distributions=param_dist_lgbm,
        n_iter=25,
        cv=5,
        scoring='accuracy',
        random_state=42,
        n_jobs=-1,
        verbose=0
    )
    random_lgbm.fit(X_train, y_train)
    show_results("LightGBM", random_lgbm, X_test, y_test, "RandomizedSearchCV")
else:
    print(" LightGBM not installed. Run: pip install lightgbm")

---
## 8. 🔵 K-Means Clustering — Elbow + Silhouette Search
K-Means is **unsupervised** — no labels, so GridSearchCV doesn't apply.
We instead search over key hyperparameters using **Elbow method** (inertia) and **Silhouette Score**.

In [ ]:
from sklearn.preprocessing import StandardScaler

X_km = StandardScaler().fit_transform(X)

# Search grid for K-Means
k_values    = range(2, 11)
init_methods = ['k-means++', 'random']
algorithms  = ['lloyd', 'elkan']

results = []

for k in k_values:
    for init in init_methods:
        for algo in algorithms:
            km = KMeans(
                n_clusters=k,
                init=init,
                algorithm=algo,
                n_init=10,
                random_state=42
            )
            labels = km.fit_predict(X_km)
            inertia = km.inertia_
            sil = silhouette_score(X_km, labels)
            results.append({
                'n_clusters': k,
                'init': init,
                'algorithm': algo,
                'inertia': round(inertia, 2),
                'silhouette_score': round(sil, 4)
            })

km_df = pd.DataFrame(results).sort_values('silhouette_score', ascending=False).reset_index(drop=True)

best = km_df.iloc[0]
print(f"\n✅ Best K-Means Configuration (by Silhouette Score):")
print(f"   n_clusters  : {int(best.n_clusters)}")
print(f"   init        : {best.init}")
print(f"   algorithm   : {best.algorithm}")
print(f"   inertia     : {best.inertia}")
print(f"   silhouette  : {best.silhouette_score}\n")

print("K-Means Hyperparameter Search Results:")
print(km_df.to_string(index=False))

---
## 📊 Final Summary — Best Hyperparameters Across All Models

In [ ]:
search_objects = [
    ("Ridge Regression",          grid_lr,     X_test_scaled, y_test),
    ("Decision Tree",             grid_dt,     X_test,        y_test),
    ("Random Forest",             random_rf,   X_test,        y_test),
    ("SVM",                       grid_svm,    X_test_scaled, y_test),
    ("MLP Neural Network",        random_mlp,  X_test_scaled, y_test),
    ("KNN",                       grid_knn,    X_test_scaled, y_test),
    ("Gradient Boosting (sklearn)",random_gb,  X_test,        y_test),
]

if XGBOOST_AVAILABLE:
    search_objects.append(("XGBoost", random_xgb, X_test, y_test))
if LIGHTGBM_AVAILABLE:
    search_objects.append(("LightGBM", random_lgbm, X_test, y_test))

summary_rows = []
for name, search, Xt, yt in search_objects:
    summary_rows.append({
        'Model': name,
        'Search Method': type(search).__name__,
        'Best CV Score': round(search.best_score_, 4),
        'Test Accuracy': round(search.best_estimator_.score(Xt, yt), 4),
        'Best Params': str(search.best_params_)
    })

# Add KMeans separately
summary_rows.append({
    'Model': 'K-Means',
    'Search Method': 'Manual Loop (Silhouette)',
    'Best CV Score': '-',
    'Test Accuracy': f"Silhouette: {km_df.iloc[0]['silhouette_score']}",
    'Best Params': str(km_df.iloc[0][['n_clusters','init','algorithm']].to_dict())
})

summary_df = pd.DataFrame(summary_rows)

print("Best Hyperparameters Found via CV Search")
print(summary_df.to_string(index=False))